# Interactive YNet L2-input viewer

The left panel overlays the measured input, the YNet nuisance prediction, and the L2 input. The right panel shows the cleaned difference `input - predicted nuisance`. The original lipid mask is used and nothing is written to disk.

In [ ]:
%matplotlib widget
from pathlib import Path

import sys

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
import numpy as np
import torch

from walinet.inference.inference import runNNLipRemoval2
from walinet.paper_figures.mask_degradation.pipeline import _compute_operator
from walinet.visualization.EvaluateModel import (
    interactive_spectra_viewer_multi,
    make_ppm_axis,
)

# ---- Settings -------------------------------------------------------------
scanner = "3T"
subject = "B0corrected_wo_LipidMask/ClimaX_Brisbane/Vol4/Res50x50"
model_name = "3T_Final"
checkpoint = "model_last.pt"
gpu_index = 0
batch_size = 200
operator_target = 0.98

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
data_root = repo_root / "data"
model_root = repo_root / "models"
subject_dir = data_root / scanner / subject
device = torch.device(f"cuda:{gpu_index}" if torch.cuda.is_available() else "cpu")
print("Subject:", subject_dir)
print("Model:", model_root / model_name)
print("Device:", device)

In [ ]:
fid = np.load(subject_dir / "OriginalData/data.npy")
magnitude = np.load(subject_dir / "OriginalData/magnitude.npy")
brain_mask = np.load(subject_dir / "masks/brain_mask.npy") > 0
lipid_mask = np.load(subject_dir / "masks/lipid_mask.npy") > 0
head_mask = brain_mask | lipid_mask
spectra = np.fft.fftshift(np.fft.fft(fid, axis=-1), axes=-1).astype(np.complex64)
del fid
n_timepoints = spectra.shape[-1]

operator, original_count, _, _ = _compute_operator(
    subject_dir=subject_dir,
    input_data="OriginalData/data.npy",
    brain_mask_path="masks/brain_mask.npy",
    lipid_mask_path="masks/lipid_mask.npy",
    n_timepoints=n_timepoints,
    n_voxels=0,
    device=device,
    target=operator_target,
)
print(f"Original lipid mask: {original_count} voxels")

cleaned = runNNLipRemoval2(
    device=device,
    exp=model_name,
    spectra=spectra,
    LipidProj_Operator_ff=operator,
    headmask=head_mask,
    batch_size=batch_size,
    normalization="auto",
    architecture="unet",
    model_root=model_root,
    checkpoint=checkpoint,
).astype(np.complex64, copy=False)
print("Inference complete.")

In [ ]:
ppm = make_ppm_axis(
    vec_size=n_timepoints,
    dwelltime=1060000,
    larmor_freq=123231706,
    ppm_ref=4.7,
    ref_freq_hz=0.0,
    dwelltime_unit="ns",
)

def spectrum_func(volume):
    return lambda x, y, z: np.real(volume[x, y, z, :])

def nuisance_func(x, y, z):
    return np.real(spectra[x, y, z, :] - cleaned[x, y, z, :])

def l2_input_func(x, y, z):
    # Unnormalized second spectral input; calculated only for the selected voxel.
    return np.real(spectra[x, y, z, :] @ operator)

def input_minus_l2_func(x, y, z):
    spectrum = spectra[x, y, z, :]
    return np.real(spectrum - spectrum @ operator)

interactive_spectra_viewer_multi(
    image_volume=magnitude,
    spectrum_funcs=[
        spectrum_func(spectra),
        nuisance_func,
        spectrum_func(cleaned),
    ],
    labels=["Input, predicted nuisance, and L2 input", "Clean-spectrum comparison"],
    overlay_labels=["Input", "Predicted nuisance"],
    extra_spectrum_funcs=[[l2_input_func], [input_minus_l2_func]],
    extra_labels=[["L2 input"], ["Input - L2 input"]],
    x_axis=ppm,
    x_label="ppm",
    ppm_min=0,
    ppm_max=4,
    compact=True,
    metabolite_lines={
        "water": 4.7, "Cr": 3.02, "GABA": 2.28, "GPC": 3.23,
        "GSH": 2.95, "Gln": 2.45, "Glu": 2.35, "Ins": 3.56,
        "NAA": 2.01, "NAAG": 2.04, "PCh": 3.22, "PCr": 3.93,
    },
)